|<img style="float:left;" src="http://pierreproulx.espaceweb.usherbrooke.ca/images/usherb_transp.gif"> |Pierre Proulx, ing, professeur|
|:---|:---|
|Département de génie chimique et de génie biotechnologique |** GCH200-Phénomènes d'échanges I **|


#### Section 6.2, exemple 6.2-2

 <img src="http://pierreproulx.espaceweb.usherbrooke.ca/Chap6-ex622.png" >
 
 <img src="http://pierreproulx.espaceweb.usherbrooke.ca/Chap6-fig622.png" >

Dans cet exemple on utilise une technique numérique au lieu de la méthode graphique d'essai et erreur proposée par Bird. On utilise toujours la solution des deux équations de f en pour trouver la vitesse avec la fonction fsolve de scipy. Cette méthode est directe et ne nécessite pas de manipulations autres que de soustraire l'équation générale de f définie en 6.1-4 de l'équation correspondant au régime de l'écoulement (laminaire (6.2-11), turbulent jusqu'à Re=40,000 (6.2-12) et turbulent à plus de 40,000 (6.2-15)).

#### Algorithme de solution:

Résoudre $\frac {1}{4} \frac {D}{L} \frac {\mathscr{P}_0 - \mathscr{P}_L} {\frac {1}{2} \rho { v_z }^2} - \frac {16}{Re}=0$ pour  $v_z$
si $\frac {\rho v_z D}{\mu} < 2100$, c'est fini, sinon

>  Résoudre $\frac {1}{4} \frac {D}{L} \frac {\mathscr{P}_0 - \mathscr{P}_L} {\frac {1}{2} \rho { v_z }^2} - \frac {0.0791}{Re^{0.25}}=0$ pour $v_z$
> si $\frac {\rho v_z D}{\mu} < 40000$, c'est fini, sinon

>> Résoudre $\frac {1}{4} \frac {D}{L} \frac {\mathscr{P}_0 - \mathscr{P}_L} {\frac {1}{2} \rho { v_z }^2} 
- [-3.6 log_{10} ( \frac {6.9}{Re} + (\frac {k/D}{3.7})^{10/9}]^{-2}=0$ pour $v_z$



In [1]:
#
# Pierre Proulx
#
# Préparation de l'affichage et des outils de calcul symbolique
#
import thermo as th
from math import *
from fluids.units import *
from thermo.units import Stream 
import numpy as np 
from scipy.optimize import fsolve, root # C'est le solver de scipy, un peu comme dans 'excel'
#
# définir la fonction dont on cherche les zéros
#
def f(vz):
    return 1/4*(D/L)*(dP/(1/2*rho*vz**2))              # équation définissant le facteur de friction par un bilan de forces équation 6.1-4
def Re(vz):
    return rho*vz*D/mu
def fLam(vz):
    fL=16/Re(vz)                                       # équation 6.2-11
    return fL-f(vz) 
def fTur(vz):
    fT=0.0791/Re(vz)**0.25                            # équation 6.2-12
    return fT-f(vz)
def fTurH(vz):
    fT=-3.6*log10(6.9/Re(vz)+(kSD/3.7)**(10/9))
    fT=(1/fT)**2                                     # équation 6.2-15
    return fT-f(vz)

In [3]:
#
# J'utiliserai fluids.units pour effectuer les conversions d'unités
#
rho=(62.4*u.lb/u.ft**3).to(u.kg/u.m**3)
mu=(1*u.cP).to(u.Pa*u.sec)
D=(7.981*u.inch).to(u.m)
dP=(1*u.lbf/u.inch**2).to(u.Pa)
L=(1000*u.ft).to(u.m)
#
# Cependant, le package fsolve ne sait pas travailler avec les unités, donc j'enlève la partie unités
# des variables qui seront utilisées dans fsolve avec la fonction magnitude (grandeur)
#
rho,mu,D,dP,L=rho.magnitude,mu.magnitude,D.magnitude,dP.magnitude,L.magnitude
aire=D**2*pi/4
kSD=2.3e-4
#
# a priori on ne sait pas si c'est laminaire ou pas,
# donc en premier on teste laminaire
#
vz=fsolve(fLam,1e-10)    # je mets une petite vitesse comme essai, cette valeur peut avoir une influence!
if Re(vz)<2100:
    print('Régime laminaire: Reynolds =  ',Re(vz),' débit massique=',rho*vz*aire,' kg/sec' )
if Re(vz)>2100:
    vz=fsolve(fTur,1e-10)
    if Re(vz) < 40000:
        print('Régime de Blasius: Reynolds =  ',Re(vz),' débit massique=',rho*vz*aire,' kg/sec')
if Re(vz)>40000:
    vz=fsolve(fTurH,1e-10)
    print('Régime de Haaland: Reynolds = ',Re(vz),' débit massique=' ,rho*vz*aire,' kg/sec' )

Régime de Haaland: Reynolds =  [145458.1338618]  débit massique= [23.15895295]  kg/sec
